# Execution Plan: Task 1 - Algorithm Development

Lightweight rule-based classification for time-series accelerometer data.

**Dataset Parameters:**
* Sampling Rate: 50 Hz
* Accelerometer Columns: ax_g, ay_g, az_g
* Target Classes: Walking, Drinking, Throwing

In [ ]:
import json
import pandas as pd
import numpy as np
from scipy import signal
from scipy.stats import kurtosis
import plotly.express as px
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import plotly.graph_objects as go
import os
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

In [ ]:
if os.path.exists("plots") == False:
    os.mkdir("plots")

In [ ]:
# Constants
SAMPLING_RATE = 50  # Hz
STEP_MS = 1000 // SAMPLING_RATE  # 20 ms
ACCEL_COLUMNS = ['ax_g', 'ay_g', 'az_g']
TARGET_CLASSES = ['Walking', 'Drinking', 'Throwing', 'Other']

# Windowing parameters
WINDOW_SIZE = 128  # 2.56 seconds at 50 Hz
OVERLAP = 0.5      # 50% overlap
STEP_SIZE = int(WINDOW_SIZE * (1 - OVERLAP)) # 64 samples

# Feature bands
LF_BAND = (0.5, 3)  # Hz
HF_BAND = (3, SAMPLING_RATE / 2)  # Hz

# Data Loading
Here we load all the datasats and the corresponding labels.

In [ ]:
# Phase 1: Data Pipeline & Segmentation

def normalize_activity_label(label):
    if label is None:
        return None
    label_lower = label.lower()
    if 'walk' in label_lower:
        return 'Walking'
    if 'throw' in label_lower:
        return 'Throwing'
    if 'drink' in label_lower:
        return 'Drinking'
    return label


def load_label_segments(label_file):
    labels_df = pd.read_csv(label_file)
    row = labels_df.iloc[0]
    segments = json.loads(row['activity'])
    label_segments = []
    for segment in segments:
        label = normalize_activity_label(segment['timeserieslabels'][0] if segment.get('timeserieslabels') else None)
        label_segments.append({
            'start': int(segment['start']),
            'end': int(segment['end']),
            'label': label,
        })
    return label_segments


def assign_activity_labels(df, segments):
    df = df.copy()
    df['activity'] = None
    for segment in segments:
        mask = (df['elapsed_ms'] >= segment['start']) & (df['elapsed_ms'] <= segment['end'])
        df.loc[mask, 'activity'] = segment['label']
    return df


def load_annotated_dataset():
    data_dir = './data/'
    sensor_files = [f'walk_throw_drink_{i}.csv' for i in range(6)]
    label_files = [f'labels_{i}.csv' for i in range(6)]

    dfs = []
    index = -1
    for sensor_file, label_file in zip(sensor_files, label_files):

        index += 1
        if index == 1 or index == 2:
            continue

        sensor_path = data_dir + sensor_file
        label_path = data_dir + label_file

        df = pd.read_csv(sensor_path)
        segments = load_label_segments(label_path)
        df = assign_activity_labels(df, segments)
        df = df.reset_index(drop=True) # Kept noisy data
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


df = load_annotated_dataset()
print(f"Loaded dataset with {len(df)} labeled samples")
print(df[['elapsed_ms', 'ax_g', 'ay_g', 'az_g', 'activity']].head())


# Windowing
Here we extract the windows from the data. Therefore we slide the windows over the data. For each window we check the labels and check which lables were assigned to the current window. In case that multiple labels are part of the current window we have rule based approach that decides if the current window is ambigous and should be not classified ("other") or which activity is the dominant one.

In [ ]:
def resample_and_interpolate(df):
    """
    Fixes Flaw 3: Enforces a strict, uninterrupted 50Hz time grid.
    If the M5 watch drops packets, this reconstructs the missing samples safely 
    so your frequency features (like LF/HF ratio) don't get corrupted.
    """
    # Sort strictly by time to prevent time-travel anomalies
    df = df.sort_values('elapsed_ms').copy()
    
    # -------------------------------------------------------------------
    # Pandas cannot resample if multiple rows claim to happen at the exact same millisecond.
    df = df.drop_duplicates(subset=['elapsed_ms'], keep='first')
    # -------------------------------------------------------------------
    
    # Convert integer ms to a proper timedelta index for Pandas resampling
    df['time_td'] = pd.to_timedelta(df['elapsed_ms'], unit='ms')
    df = df.set_index('time_td')
    
    # 1. Resample sensor data: linear interpolation for missing packets
    numeric_resampled = df[ACCEL_COLUMNS].resample(f'{STEP_MS}ms').mean().interpolate(method='linear')
    

    # 2. Resample labels: forward-fill with a TINY limit (e.g., limit=2) 
    # This just snaps the label to the 50Hz grid without bleeding into true empty space.
    labels_resampled = df[['activity']].resample(f'{STEP_MS}ms').ffill(limit=2)
    
    # 3. Explicitly mark all the remaining empty space as 'Other'
    labels_resampled['activity'] = labels_resampled['activity'].fillna('Other')    

    # Recombine
    final_df = numeric_resampled.join(labels_resampled)
    
    # Drop rows where we couldn't interpolate accelerometer data (e.g., massive gaps)
    final_df = final_df.dropna(subset=ACCEL_COLUMNS).reset_index(drop=True)
    return final_df

def priority_majority_vote(label_window):
    """
    Fixes Flaws 1 & 2: Prioritizes brief, explosive movements.
    """
    total_samples = len(label_window)
    
    # Count occurrences of valid string labels (ignoring NaNs/Nulls)
    valid_labels = [str(l) for l in label_window if pd.notna(l) and l != 'nan']
    if not valid_labels:
        return 'Other'
        
    counts = pd.Series(valid_labels).value_counts()
    
    # Calculate proportion of the ENTIRE window (including null space)
    proportions = counts / total_samples
    
    # RULES TO DECIDE HOW A WINDOW SHOULD BE CLASSIFIED DEPENDING ON LABEL PARTS
    if 'Throwing' in proportions and proportions['Throwing'] >= 0.3:
        return 'Throwing'
        
    if 'Drinking' in proportions and proportions['Drinking'] >= 0.45:
        return 'Drinking'
        
    if 'Walking' in proportions and proportions['Walking'] > 0.70:
        return 'Walking'
        
    # If no criteria is met (e.g., window is 90% Null), drop the window
    return 'Other'

def create_robust_windows(df):
    """
    Fixes Flaw 4: Uses efficient memory views to slice data instantly.
    """
    windows_out = []
    
    # Process each recording independently to avoid stitching different files together
    groups = [group for _, group in df.groupby('recording_id')] if 'recording_id' in df.columns else [df]
    
    for group in groups:
        # First, ensure continuous 50Hz grid
        group_resampled = resample_and_interpolate(group)
        
        accel_data = group_resampled[ACCEL_COLUMNS].values
        labels = group_resampled['activity'].values
        
        if len(accel_data) < WINDOW_SIZE:
            continue
            
        # Create sliding windows instantly using Numpy C-bindings
        # sliding_window_view creates shape: (num_windows, num_columns, window_size)
        raw_accel_windows = sliding_window_view(accel_data, WINDOW_SIZE, axis=0)[::STEP_SIZE]
        raw_label_windows = sliding_window_view(labels, WINDOW_SIZE, axis=0)[::STEP_SIZE]
        
        # Swap axes so accel arrays are shaped (128, 3) to match your feature extractor logic
        accel_windows = np.swapaxes(raw_accel_windows, 1, 2)
        
        for i in range(len(raw_label_windows)):
            assigned_label = priority_majority_vote(raw_label_windows[i])
            
            # Only keep windows that successfully passed the priority vote
            if assigned_label is not None:
                windows_out.append((accel_windows[i], assigned_label))
                
    return windows_out

# Execute the robust pipeline
windows = create_robust_windows(df)
print(f"Created {len(windows)} robust windows ready for feature extraction.")
from collections import Counter
activity_counts = Counter(label for _, label in windows)
print("Window counts per activity:")
for activity, count in activity_counts.items():
    print(f"{activity}: {count}")


# Feature Computation
Here we calculate the features that seemed promising in the previous Milestone.

In [ ]:
# Phase 2: Lean Feature Extraction

def compute_features(window_df, squared_magnitude=False):
    """
    Compute all available features from a window of accelerometer data.
    Returns dict with V1 and V2 feature sets.
    """
    if isinstance(window_df, np.ndarray):
        window_df = pd.DataFrame(window_df, columns=ACCEL_COLUMNS)
    elif not isinstance(window_df, pd.DataFrame):
        window_df = pd.DataFrame(window_df, columns=ACCEL_COLUMNS)

    # V1 features: magnitude-based descriptors
    if squared_magnitude:
        magnitude = np.sum(window_df.values**2, axis=1)
    else:
        magnitude = np.sqrt(np.sum(window_df.values**2, axis=1))
    jerk = np.diff(magnitude) / (1 / SAMPLING_RATE)
    max_abs_jerk = np.max(np.abs(jerk)) if len(jerk) > 0 else 0
    peak_to_peak = np.max(magnitude) - np.min(magnitude)
    freqs, psd = signal.welch(magnitude, fs=SAMPLING_RATE, nperseg=min(256, len(magnitude)))
    lf_mask = (freqs >= LF_BAND[0]) & (freqs <= LF_BAND[1])
    hf_mask = (freqs >= HF_BAND[0]) & (freqs <= HF_BAND[1])
    lf_power = np.sum(psd[lf_mask])
    hf_power = np.sum(psd[hf_mask])
    lf_hf_ratio = lf_power / hf_power if hf_power > 0 else 0

    # V2 features: axis-specific posture and peak-to-peak
    postural_mean = window_df['az_g'].mean()
    postural_ptp = window_df['az_g'].max() - window_df['az_g'].min()

    # New features
    signal_variance = float(np.var(magnitude))
    mag_centered = magnitude - np.mean(magnitude)
    zero_crossing_rate = int(((mag_centered[:-1] * mag_centered[1:]) < 0).sum())
    kurtosis_val = float(kurtosis(magnitude))
    mean_x = float(window_df['ax_g'].mean())
    mean_y = float(window_df['ay_g'].mean())
    mean_z = float(window_df['az_g'].mean())
    corr_zx_val = window_df['az_g'].corr(window_df['ax_g'])
    corr_zx = float(corr_zx_val) if pd.notna(corr_zx_val) else 0.0
    # Autocorrelation of magnitude to find periodicity
    autocorr = signal.correlate(mag_centered, mag_centered, mode='full')
    autocorr = autocorr[len(mag_centered)-1:] # positive lags
    if len(autocorr) > 1:
        max_corr = float(np.max(autocorr[1:]))
        lag = int(np.argmax(autocorr[1:]) + 1)
        max_corr = float(max_corr / autocorr[0]) if autocorr[0] > 0 else 0.0
    else:
        max_corr = 0.0
        lag = 0

    return {
        'max_abs_jerk': max_abs_jerk,
        'peak_to_peak': peak_to_peak,
        'lf_hf_ratio': lf_hf_ratio,
        'postural_mean': postural_mean,
        'postural_ptp': postural_ptp,
        'signal_variance': signal_variance,
        'zero_crossing_rate': zero_crossing_rate,
        'kurtosis': kurtosis_val,
        'max_correlation': max_corr,
        'lag': lag,
        'mean_x': mean_x,
        'mean_y': mean_y,
        'mean_z': mean_z,
        'corr_zx': corr_zx
    }

# Visualization
The following section allows you to inspect the different featues and see if they have classification potential. You can select 3 features which are then displayed in a 3D grid and afterwards in 2D grids which visualize different combinatinos of the selected features.

In [ ]:
# Build feature DataFrame
features_list = []
for window_data, label in windows:
    features = compute_features(window_data, squared_magnitude=False)
    features['activity'] = label
    features_list.append(features)

features_df = pd.DataFrame(features_list)
print(f"Feature DataFrame shape: {features_df.shape}")
print(features_df.head())

# Select the final three features to visualize / compare manually.
FINAL_FEATURE_SET = ['max_abs_jerk', 'corr_zx', 'kurtosis']

FEATURE_LABELS = {
    'max_abs_jerk': 'Max Absolute Jerk',
    'peak_to_peak': 'Peak-to-Peak Range',
    'lf_hf_ratio': 'LF/HF Energy Ratio', # NO -TOO EXPENSIVE
    'postural_mean': 'Postural Mean',  # no
    'postural_ptp': 'Postural PTP',  # no
    'signal_variance': 'Signal Variance', # no
    'zero_crossing_rate': 'Zero Crossing Rate',  # no
    'kurtosis': 'Kurtosis',
    'max_correlation': 'Max Correlation',  # NO -TOO EXPENSIVE
    'lag': 'Lag',  # no,
    'mean_x': 'Mean X',
    'mean_y': 'Mean Y',
    'mean_z': 'Mean Z',
    'corr_zx': 'Pearson Corr Z-X'
}

In [ ]:
# Generate 3D scatter plot
x_feat, y_feat, z_feat = FINAL_FEATURE_SET
fig = px.scatter_3d(
    features_df,
    x=x_feat,
    y=y_feat,
    z=z_feat,
    color='activity',
    title=f"Feature Space: {FEATURE_LABELS[x_feat]} vs {FEATURE_LABELS[y_feat]} vs {FEATURE_LABELS[z_feat]}",
    labels={
        x_feat: FEATURE_LABELS.get(x_feat, x_feat),
        y_feat: FEATURE_LABELS.get(y_feat, y_feat),
        z_feat: FEATURE_LABELS.get(z_feat, z_feat)
    }
)

fig.show()

In [ ]:

def plot_2d_feature_combinations(df):
    """
    Generates a 2D scatter matrix for all feature combinations.
    This makes it much easier to visually identify linear thresholds.
    """
    fig = px.scatter_matrix(
        df,
        dimensions=FINAL_FEATURE_SET,
        color='activity',
        title='2D Feature Projections (Scatter Matrix)',
        labels={feat: FEATURE_LABELS.get(feat, feat) for feat in FINAL_FEATURE_SET},
        opacity=0.7 # Slight transparency helps show overlapping clusters
    )
        
    # Hide the diagonal (which just compares a feature to itself)
    fig.update_traces(diagonal_visible=False)
        
    # Make the plot large enough to read easily
    fig.update_layout(width=1000, height=800)
    fig.write_image("plots/feature_scatter_matrix.png")
    fig.show()

# Call the function using your existing features_df
plot_2d_feature_combinations(features_df)

# Find optimal Threshholds through Machine Learning
Since manually testing thresshold yielded very bad results, we used machine learning algorithms to find the optimal splits.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from micromlgen import port

# 1. Define your lightweight feature columns
# (Ensure these match what you compute on the watch)
FEATURES = ['mean_x', 'mean_y', 'mean_z', 'signal_variance', 'max_abs_jerk', 'peak_to_peak', 'kurtosis', 'zero_crossing_rate', 'postural_mean', 'corr_zx']
TARGET = 'activity'

X = features_df[FEATURES]
y = features_df[TARGET]

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the Decision Tree
# CRITICAL: Keep max_depth low (3 to 5). 
# A deep tree overfits and creates massive C++ files.
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)

# 4. Evaluate the optimal thresholds
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

# 5. Export to pure C++
# This generates a raw string of C++ code you can paste into Arduino IDE / PlatformIO
c_code = port(clf, classmap={
    0: "Drinking", 
    1: "Other", 
    2: "Throwing", 
    3: "Walking"
})

# print("\n--- C++ Code for M5 Watch ---")
# print(c_code)

In [ ]:
# 1. Create a large canvas so the text is readable
plt.figure(figsize=(20, 12))

# 2. Plot the tree
plot_tree(clf, 
          feature_names=FEATURES, 
          class_names=["Drinking", "Other", "Throwing", "Walking"], # Must match your classmap
          filled=True,      # Colors the nodes based on the dominant class
          rounded=True,     # Makes it look slightly nicer
          fontsize=12,
          proportion=False) # Set to True if you want percentages instead of raw counts

plt.title("Decision Tree Rules (C++ Logic Blueprint)", fontsize=16)
plt.tight_layout()

# 3. Save it to your existing plots folder
plt.savefig("plots/decision_tree_visualization.png", dpi=300)
plt.show()

# Classification and Evaluation
Here in this section we apply the threshholds to actually classify the windows to our acitivities. We also calculate evaluation metrics.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import plotly.graph_objects as go
import os

def evaluate_model(y_true, y_pred, classes, title, filename):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Accuracy: {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1 Score: {f1:.3f}")

    cm = confusion_matrix(y_true, y_pred, labels=classes)
    print("\nConfusion Matrix:")
    print(cm)

    fig_cm = go.Figure(data=go.Heatmap(
        z=cm,
        x=classes,
        y=classes,
        colorscale='Blues',
        text=cm,
        texttemplate="%{text}",
        textfont={"size":12}
    ))

    fig_cm.update_layout(
        title=title,
        xaxis_title='Predicted',
        yaxis_title='True'
    )

    if not os.path.exists("plots"):
        os.mkdir("plots")
        
    filepath = os.path.join("plots", filename)
    fig_cm.write_image(filepath)
    print(f"Plot successfully saved to: {os.path.abspath(filepath)}")
    fig_cm.show()

# 1. Generate predictions instantly using the trained model
y_pred = clf.predict(X_test)
y_true = y_test

# 2. Evaluate
evaluate_model(y_true, y_pred, TARGET_CLASSES, 'Decision Tree Confusion Matrix (Test Set)', 'dt_confusion_matrix.png')


# Binary Classification: Throwing vs Noise
In this section, we reduce the problem to a binary classification task to optimize specifically for detecting throws. Everything that is not a throw is considered noise.

In [ ]:
# 1. Binarize the target
y_train_bin = y_train.map(lambda x: 'Throwing' if x == 'Throwing' else 'Noise')
y_test_bin = y_test.map(lambda x: 'Throwing' if x == 'Throwing' else 'Noise')

# Adapt features
FEATURES = ['mean_x', 'mean_y', 'mean_z', 'signal_variance', 'max_abs_jerk', 'peak_to_peak', 'kurtosis']
X = features_df[FEATURES]
y = features_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Train the Decision Tree
clf_bin = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_bin.fit(X_train, y_train_bin)

# 3. Export to C++
c_code_bin = port(clf_bin, classmap={0: "Noise", 1: "Throwing"})
# print(c_code_bin)

# 4. Visualize the tree
plt.figure(figsize=(20, 12))
plot_tree(clf_bin, 
          feature_names=FEATURES, 
          class_names=["Noise", "Throwing"], 
          filled=True,      
          rounded=True,     
          fontsize=12,
          proportion=False)

plt.title("Decision Tree Rules (Throwing vs Noise)", fontsize=16)
plt.tight_layout()
plt.savefig("plots/decision_tree_bin_visualization.png", dpi=300)
plt.show()


## Classification and Evaluation (Binary)
Evaluate the binary classification model using the same metrics as before.

In [ ]:
# 1. Generate predictions
y_pred_bin = clf_bin.predict(X_test)
y_true_bin = y_test_bin

# 2. Evaluate using the previously defined function
evaluate_model(y_true_bin, y_pred_bin, ['Noise', 'Throwing'], 'Binary Decision Tree Confusion Matrix', 'dt_bin_confusion_matrix.png')

# Final Model
For actual usage we **retrain** our model on full dataset (dont exclude test here).

In [ ]:
# Create the binary target for the FULL dataset
y_bin = y.map(lambda x: 'Throwing' if x == 'Throwing' else 'Noise')

# Retrain on the entire dataset for deployment
final_clf_bin = DecisionTreeClassifier(max_depth=4, random_state=42)
final_clf_bin.fit(X, y_bin) 

# Export to C++
c_code_bin = port(final_clf_bin, classmap={0: "Noise", 1: "Throwing"})


In [ ]:
print(c_code_bin)

In [ ]:
# Save C code to file
with open('final_model.h', 'w') as f:
    f.write(c_code_bin)

print("C code saved to 'final_model.h'")

# Computed Throw Statistics 
For our detected throws we will send the window to a server for further analysis. This analysis include the following features:
- Max Peak Acceleration
- Drift

The final product should include additional trajectory analysis, however this would require additional gyroscope data, which this current version of the dataset does not possess.

In [ ]:
# Predict on the entire dataset to extract all identified throws
y_pred_all = final_clf_bin.predict(X)

# Get the indices where the model predicted 'Throwing'
# X has the same index as features_df, and the 'windows' list has the same order.
predicted_throw_indices = np.where(y_pred_all == 'Throwing')[0]

# Extract the raw window data (e.g., the 128x3 arrays of accelerometer data)
detected_throw_windows = [windows[i][0] for i in predicted_throw_indices]

# Also extract their true labels in case we want to analyze true vs false positives
detected_throw_true_labels = [windows[i][1] for i in predicted_throw_indices]

print(f"Total windows processed: {len(windows)}")
print(f"Number of windows classified as 'Throwing': {len(detected_throw_windows)}")

# The raw data for the first detected throw can be accessed like this:
# print(detected_throw_windows[0].shape)


#### Metric 1: Max Peak Acceleration

In [ ]:
def analyze_peak_acceleration():
    """
    Analyze peak acceleration statistics for all detected throws.
    """
    peak_accelerations = []
    
    # Extract peak acceleration for each detected throw window
    for window in detected_throw_windows:
        # Calculate magnitude of acceleration for each sample
        magnitude = np.sqrt(np.sum(window**2, axis=1))
        peak_accel = np.max(magnitude)
        peak_accelerations.append(peak_accel)
    
    peak_accelerations = np.array(peak_accelerations)
    
    # Calculate statistics
    max_peak = np.max(peak_accelerations)
    mean_peak = np.mean(peak_accelerations)
    std_peak = np.std(peak_accelerations)
    
    # Print results
    print(f"Peak-acceleration in throws (max across train segments): {max_peak:.3f}")
    print(f"Mean peak-acceleration over {len(detected_throw_windows)} throw segments: {mean_peak:.3f}")
    print(f"Standard deviation of Peak-acceleration: {std_peak:.3f}")
    
    return {
        'max_peak': max_peak,
        'mean_peak': mean_peak,
        'std_peak': std_peak,
        'num_throws': len(detected_throw_windows)
    }

# Execute the analysis
throw_stats = analyze_peak_acceleration()

The peak-acceleration describes the highest impulse strength during the throw. A high peak indicates an explosive, ballistic movement; a lower peak suggests a more controlled throw. These train-set statistics can be used to set thresholds for deciding whether a throw is controlled or not.

In [ ]:
import random

def categorize_throws_by_control(detected_throw_windows, throw_stats):
    """
    Categorize detected throws into control levels based on peak acceleration.
    Uses mean and standard deviation to define thresholds.
    """
    # Calculate thresholds based on statistics
    mean_peak = throw_stats['mean_peak']
    std_peak = throw_stats['std_peak']
    
    # Define thresholds: mean ± 0.5*std
    controlled_threshold = mean_peak - 0.5 * std_peak
    uncontrolled_threshold = mean_peak + 0.5 * std_peak
    
    print("=" * 60)
    print("Categorization of Detected Throws by Control Level")
    print("=" * 60)
    print(f"\nCalculated Thresholds:")
    print(f"  Controlled threshold:        {controlled_threshold:.3f} g")
    print(f"  Uncontrolled threshold:      {uncontrolled_threshold:.3f} g")
    print(f"\nThreshold Determination Method:")
    print(f"  Thresholds were calculated using mean ± 0.5*std of peak acceleration across all detected throws. This divides throws into 3 roughly balanced categories.")
    print(f"\nStatistics used:")
    print(f"  Mean peak acceleration:      {mean_peak:.3f} g")
    print(f"  Std peak acceleration:       {std_peak:.3f} g")
    print("=" * 60)

   
    # Categorize all throws
    throw_categories = []
    for window in detected_throw_windows:
        magnitude = np.sqrt(np.sum(window**2, axis=1))
        peak_accel = np.max(magnitude)
        
        if peak_accel <= controlled_threshold:
            category = 'Controlled'
        elif peak_accel <= uncontrolled_threshold:
            category = 'Moderately Controlled'
        else:
            category = 'Uncontrolled'
        
        throw_categories.append({
            'peak_acceleration': peak_accel,
            'category': category
        })

    # Sample 10 random throws and display them
    sample_indices = random.sample(range(len(throw_categories)), min(5, len(throw_categories)))
    
    print(f"\nSample of 5 Random Throws:\n")
    print(f"{'Index':<8} {'Peak Accel (g)':<18} {'Category':<22}")
    print("-" * 48)
    
    for i, idx in enumerate(sorted(sample_indices), 1):
        throw = throw_categories[idx]
        print(f"{idx:<8} {throw['peak_acceleration']:<18.3f} {throw['category']:<22}")
    
    print("=" * 60)

    # throw counts & percentage for summary
    total_throws = len(throw_categories)
    counts = Counter(tc['category'] for tc in throw_categories)
    
    print("\n")
    print("=" * 60)
    print("Summary of Throw Control Categories")
    print("=" * 60)

    print(f"Total throws: {total_throws}")
    for category in ['Controlled', 'Moderately Controlled', 'Uncontrolled']:
        count = counts.get(category, 0)
        percentage = (count / total_throws * 100) if total_throws else 0.0
        print(f"{category}: {count} ({percentage:.1f}%)")
    
    return throw_categories

# Execute the categorization
throw_categories = categorize_throws_by_control(detected_throw_windows, throw_stats)

#### Metric 2: Drift within a Throw
The metric analyzes the directional stability of throws by dividing each throw into several time intervals and calculating the average direction of movement in each interval. It then determines the angle between the initial and final directions of each throw to measure the change in direction (“drift”) and evaluate it statistically.

In [ ]:
def compute_window_directions(ax, ay, az, n_windows=5):
    ax = np.asarray(ax)
    ay = np.asarray(ay)
    az = np.asarray(az)

    if len(ax) < n_windows:
        return []

    stacked = np.vstack([ax, ay, az]).T
    windows_split = np.array_split(stacked, n_windows)

    directions = []
    for w in windows_split:
        mean_vec = np.mean(w, axis=0)
        norm = np.linalg.norm(mean_vec)
        if norm < 1e-9:
            directions.append(np.array([0.0, 0.0, 0.0]))
        else:
            directions.append(mean_vec / norm)
    return directions

def angle_between_vectors(v1, v2):
    v1 = np.asarray(v1)
    v2 = np.asarray(v2)
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 1e-9 or n2 < 1e-9:
        return 0.0
    dot = np.dot(v1, v2) / (n1 * n2)
    dot = np.clip(dot, -1.0, 1.0)
    return float(np.degrees(np.arccos(dot)))

def compute_throw_drift_stats(throw_windows, n_windows=5):
    angles = []
    for w in throw_windows:
        if not isinstance(w, np.ndarray) or w.shape[1] < 3:
            continue
        ax, ay, az = w[:,0], w[:,1], w[:,2]
        dirs = compute_window_directions(ax, ay, az, n_windows=n_windows)
        if len(dirs) < 2:
            continue
        angles.append(angle_between_vectors(dirs[0], dirs[-1]))

    if len(angles) == 0:
        print("No valid throws to analyze.")
        return {}

    angles = np.array(angles)
    stats = {
        'max_deg': float(np.max(angles)),
        'p25_deg': float(np.percentile(angles, 25)),
        'median_deg': float(np.percentile(angles, 50)),
        'p75_deg': float(np.percentile(angles, 75)),
        'n_throws': int(len(angles))
    }

    print(f"Max drift angle: {stats['max_deg']:.2f}°")
    print(f"25th percentile: {stats['p25_deg']:.2f}°")
    print(f"Median (50th):   {stats['median_deg']:.2f}°")
    print(f"75th percentile: {stats['p75_deg']:.2f}°")

    return stats

# Run on all detected throws 
drift_stats = compute_throw_drift_stats(detected_throw_windows, n_windows=5)

In [ ]:
def classify_throw_drift(drift_angle, threshold_low, threshold_high):
    """
    Classify throw drift level based on mean ± 0.5*std thresholds.
    """
    if drift_angle <= threshold_low:
        return "Low Drift"
    elif drift_angle <= threshold_high:
        return "Moderate Drift"
    else:
        return "Strong Drift"


def categorize_throws_by_drift(throw_windows, n_windows=5):
    """
    Categorize detected throw windows into low / moderate / strong drift classes.
    Uses mean ± 0.5*std from the available throw drift angles as thresholds.
    """
    drift_results = []
    drift_angles = []

    for window_idx, window in enumerate(throw_windows):
        if not isinstance(window, np.ndarray) or window.shape[1] < 3:
            continue

        ax, ay, az = window[:, 0], window[:, 1], window[:, 2]
        directions = compute_window_directions(ax, ay, az, n_windows=n_windows)
        if len(directions) < 2:
            continue

        drift_angle = angle_between_vectors(directions[0], directions[-1])
        drift_results.append({
            'window_idx': window_idx,
            'drift_angle_deg': float(drift_angle)
        })
        drift_angles.append(drift_angle)

    if len(drift_angles) == 0:
        print("No valid throw windows available for drift classification.")
        return []

    mean_drift = float(np.mean(drift_angles))
    std_drift = float(np.std(drift_angles))

    threshold_low = mean_drift - 0.5 * std_drift
    threshold_high = mean_drift + 0.5 * std_drift

    print("=" * 60)
    print("DRIFT-BASED THROW CLASSIFICATION")
    print("=" * 60)
    print(f"\nDrift Thresholds (derived from available throws):")
    print(f"  Low Drift:       drift ≤ {threshold_low:.2f}°")
    print(f"  Moderate Drift:  {threshold_low:.2f}° < drift ≤ {threshold_high:.2f}°")
    print(f"  Strong Drift:    drift > {threshold_high:.2f}°")
    print(f"\nThreshold Determination Method:")
    print(f"  Thresholds were calculated using mean ± 0.5*std of drift angle across all detected throws.")
    print(f"\nStatistics used:")
    print(f"  Mean drift angle:           {mean_drift:.2f}°")
    print(f"  Std drift angle:            {std_drift:.2f}°")
    print("\n" + "─" * 60)

    categorized = []
    for item in drift_results:
        classification = classify_throw_drift(
            item['drift_angle_deg'],
            threshold_low,
            threshold_high
        )
        categorized.append({
            'window_idx': item['window_idx'],
            'drift_angle_deg': item['drift_angle_deg'],
            'classification': classification
        })

    print("─" * 60)

    total = len(categorized)
    counts = Counter(item['classification'] for item in categorized)
    
    sample_indices = random.sample(range(total), min(5, total))

    print(f"\nSample of {len(sample_indices)} Random Throws:")
    print(f"{'Idx':<8} {'Window':<8} {'Drift (°)':<12} {'Category':<20}")
    print("-" * 60)

    for sample_idx in sorted(sample_indices):
        item = categorized[sample_idx]
        print(
            f"{sample_idx:<8} "
            f"{item['window_idx']:<8} "
            f"{item['drift_angle_deg']:<12.2f} "
            f"{item['classification']:<20}"
        )

    print("\n")
    print("=" * 60)
    print("Summary of Drift Classifications")
    print("=" * 60)
    for label in ['Low Drift', 'Moderate Drift', 'Strong Drift']:
        count = counts.get(label, 0)
        percentage = (count / total * 100) if total else 0.0
        print(f"  {label:<15} {count}/{total} ({percentage:.1f}%)")

    return categorized

throw_drift_categories = categorize_throws_by_drift(detected_throw_windows, n_windows=5)

**Throw Analysis**
To place a stronger focus on the throws, we focused on two different metrics and the first metric is magnitude, i.e., peak acceleration. Assuming that beer pong does not rely on long throws with strong acceleration, we categorized our throws into "controlled", "moderately controlled" and "uncontrolled". In this case, throws categorized as “controlled” have a lower magnitude, while throws categorized as “uncontrolled” have a higher magnitude; for this reason, based on our understanding of beer pong, throws labelled as "uncontrolled" have greater force, which is undesirable and and it can cause the ball to fly past the target. We can see that out of 202 throws, only 43.1% got labelled as "controlled", 34.7% as "moderately controlled" and 22.3% as "uncontroleld".

We then examined the second metric: the drift within a single throw across all axes. This involves comparing the dominant direction, the acceleromenter direction vector, at the start with that at the end, allowing us to draw conclusions about how a particular throw unfolded. Based on this, we categorized the test data into “low drift,” “moderate drift,” and “strong drift.”. To put it simply, if a player's arm is positioned directly opposite the target, a “low drift” rating is considered good. We can see that 38.6% of our observed throws got labelled with "Low Drift", 35.1% with "Moderate Drift" and 26.2% with "Strong Drift".

***A critical examination of the metrics and a more in-depth approach:***
As previously mentioned, these metrics and their evaluation are based on highly simplified assumptions. We have assumed that shots with a low magnitude are better. However, a player can also hit the target with precision even with a high magnitude, for example by shooting the ball sharply upward. Accordingly, it would be possible to use more neutral labels (as with drift labels) and evaluate individually for each player which label should be targeted for their throwing style. Therefore, it would be helpful to create user profiles and store information regarding their throwing style.
A single metric alone probably says little about the throw as a whole, and we have ultimately identified only two suitable metrics here. A more reliable analysis of throws could be improved by combining several different metrics that incorporate additional data, such as gyroscope data, spatial orientation, and the size of the playing field, in addition to the used accelerometer data. In a project with a broader scope, it might even be possible to identify the flight path using additional data. However, this would specifically require the use of the gyroscope, whose data was not recorded in our case.